# SnorlaxCode: final submissions, Week 1 and Week 2

IAA AI-Assisted Actuarial Hackathon 2026. Team SnorlaxCode: Samuel Lim, Wendy Low, Benedict Tan.
Full code and write-up: our GitHub repository (see `README.md` there).

This notebook rebuilds both of our final prediction files. Week 2 comes out byte for byte the same as the file we submitted.
Week 1 equals it to about 1e-14 relative (floating-point rounding in the last digit, so the MD5 differs; the notebook requires 1e-9).

| Output in `/kaggle/working/` | Model | Public | Private |
|---|---|---|---|
| `submission_week1.csv` | Week 1 (developed market) final `w9_motor2_pre_m0.5` | 0.12965 | 0.13923 |
| `submission_week2.csv` | Week 2 (emerging market) final #31 `motorlevel_l_full` | 0.16840 | |

No models are fitted here. The model outputs were made by our full pipelines, which take hours: Week 1 `week1/final_week1_R_Script/run_all.R`
and Week 2 `week2/run/build_final_motorlevel.ps1 -Full`. They are attached as a dataset. The notebook then does the final steps:
the blend and credibility for Week 1, and the component sum and motor level review for Week 2. It uses the same arithmetic
as our quick rebuild scripts, `week1/docs/handover_final/replicate/replicate_final_submission.R` and `week2/R/30_final_from_components.R`.

Every file is checked:
- its row count;
- no row duplicated, missing or negative;
- the same row ids as the competition's sample submission;
- the same MD5 fingerprint as the submitted file, or failing that, the same values to 1e-9 relative.

**Inputs** (Add Input):
- the Part 2 competition data, which holds both the `emerging` and the `developed` sample submissions;
- our dataset `snorlaxcode-final-components`, which holds the model outputs and, for the checks, the two submitted files.

## 0. Setup

In [ ]:
suppressMessages({library(data.table); library(arrow)})
INPUT <- Sys.getenv("KAGGLE_INPUT_DIR", "/kaggle/input")      # env vars only for testing outside Kaggle
OUT   <- Sys.getenv("KAGGLE_WORKING_DIR", "/kaggle/working")

all_files <- list.files(INPUT, recursive = TRUE, full.names = TRUE)
find_one <- function(regex) {
  f <- grep(regex, all_files, value = TRUE)
  if (length(f) != 1) stop(sprintf("Expected 1 file matching '%s' under %s, found %d", regex, INPUT, length(f)))
  f
}

# Check a prediction table, write it exactly as the submitted file was written (Windows line endings), and confirm the MD5.
# If the MD5 differs (the last digit of a number can change between machines through floating-point rounding),
# the values must equal the submitted file (attached in our dataset) to a relative 1e-9.
check_and_write <- function(s, n_rows, want_md5, final_regex, file) {
  ss <- NULL
  for (f in grep("sample_submission\\.(parquet|csv)$", all_files, value = TRUE)) {
    x <- if (grepl("parquet$", f)) as.data.table(read_parquet(f)) else fread(f, colClasses = list(character = "row_id"))
    if (nrow(x) == n_rows) { ss <- x; break }
  }
  if (is.null(ss)) stop("No ", n_rows, "-row sample_submission found: add the Part 2 competition data as an input")
  ok <- c(rows = nrow(s) == n_rows, unique = !anyDuplicated(s$row_id), no_missing = !anyNA(s$prediction),
          not_negative = all(s$prediction >= 0), same_ids_as_sample = setequal(as.character(ss$row_id), s$row_id))
  if (!all(ok)) stop("Failed: ", paste(names(ok)[!ok], collapse = ", "))
  path <- file.path(OUT, file)
  fwrite(s, path, scipen = 50, eol = "\r\n")
  md5 <- unname(tools::md5sum(path))
  if (md5 == want_md5) return(cat(sprintf("OK  %s: %s rows, row ids = sample submission, MD5 %s = submitted file, byte for byte\n",
                                          file, format(n_rows, big.mark = ","), md5)))
  f_final <- grep(final_regex, all_files, value = TRUE)[1]
  if (is.na(f_final)) stop(file, ": MD5 ", md5, " differs from ", want_md5, " and the submitted file is not attached to compare values")
  cmp <- fread(f_final, colClasses = list(character = "row_id"))[s, on = "row_id"]
  rel <- max(abs(cmp$prediction - cmp$i.prediction) / pmax(cmp$prediction, 1e-9))
  if (nrow(cmp) != n_rows || anyNA(cmp$prediction) || rel > 1e-9) stop(file, ": values differ from the submitted file (max rel diff ", rel, ")")
  cat(sprintf("OK  %s: values equal the submitted file (max relative difference %.1e); last-digit floating-point rounding only, so the MD5 differs\n", file, rel))
}
line_of <- function(row_id) c(B = "buildings", C = "contents", M = "motor")[substr(row_id, 1, 1)]

## Part A. Week 1 final: `w9_motor2_pre_m0.5` (developed market)

For each line, the frozen panel holds, per policy, the per-peril predictions of two engines (`lad_*`: the GBM ladder,
`c5_*`: component C5), the SAM 3 model (`sam3`), and the policy's own claim count `N` and expected count `E`. The steps are:

- per line: `b = LW x sum_p [w_p x ladder_p + (1 - w_p) x C5_p] + (1 - LW) x SAM3` (WENDY 9 blend)
- motor: `b = 0.5 x b + 0.5 x MOTOR2 x sum(b) / sum(MOTOR2)` (MOTOR 2 spline GLM, mixed in before credibility)
- all lines: `q = b x (A + N) / (A + E)`, rescaled to `sum(b)` (WENDY 8 credibility on the policy's own claim history)

In [ ]:
PERILS <- list(motor = c("own_damage", "third_party_prop", "bodily_injury", "theft", "glass_hail"),
               buildings = c("storm", "flood", "escape_of_water", "subsidence", "fire", "accidental_damage"),
               contents = c("theft", "escape_of_water", "accidental_damage", "flood"))
W <- list(motor = c(own_damage = 0, third_party_prop = 1, bodily_injury = 0, theft = 0, glass_hail = 1),
          buildings = c(storm = 1, flood = 0, escape_of_water = 0, subsidence = 0.5, fire = 0.25, accidental_damage = 0.25),
          contents = c(theft = 0, escape_of_water = 0.75, accidental_damage = 0, flood = 0.75))
LW <- c(motor = 0.9, buildings = 0.7, contents = 0.9)
A_CRED <- c(motor = 8, buildings = 5, contents = 5)
M_MOTOR2 <- 0.5

num <- function(Q, col) {
  v <- as.numeric(Q[[col]])
  v[is.na(v)] <- 0
  v
}
cred <- function(Q, ln, b) {
  q <- b * (A_CRED[[ln]] + num(Q, "N")) / (A_CRED[[ln]] + num(Q, "E"))
  q * sum(b) / sum(q)
}
line_pred <- function(Q, ln) {
  pb <- Reduce(`+`, lapply(PERILS[[ln]], function(p) W[[ln]][[p]] * num(Q, paste0("lad_", p)) + (1 - W[[ln]][[p]]) * num(Q, paste0("c5_", p))))
  b <- LW[[ln]] * pb + (1 - LW[[ln]]) * num(Q, "sam3")
  if (ln == "motor") b <- (1 - M_MOTOR2) * b + M_MOTOR2 * Q$m2 * sum(b) / sum(Q$m2)
  cred(Q, ln, b)
}

D <- lapply(setNames(names(PERILS), names(PERILS)), function(ln) as.data.table(read_parquet(find_one(sprintf("panel_%s_2025\\.parquet$", ln)))))
m2 <- fread(find_one("motor2_only_20261023_1359\\.csv$"), colClasses = list(character = "row_id"))[, .(row_id, m2 = prediction)]
D$motor <- merge(D$motor, m2, by = "row_id", all.x = TRUE, sort = FALSE)
stopifnot(!anyNA(D$motor$m2))
w1 <- rbindlist(lapply(names(D), function(ln) data.table(row_id = D[[ln]]$row_id, prediction = line_pred(D[[ln]], ln))))
ord <- fread(find_one("row_order\\.csv$"), colClasses = list(character = "row_id"))
stopifnot(nrow(w1) == nrow(ord), setequal(w1$row_id, ord$row_id))
w1 <- w1[match(ord$row_id, row_id)]

check_and_write(w1, 626639L, "148a9f430289965107f40175dd2953ca",
                "week1_submission_20261023_1636_w9_motor2_pre_m0\\.5\\.csv$", "submission_week1.csv")
w1[, .(policies = .N, mean_prediction = round(mean(prediction), 2)), by = .(line = line_of(row_id))]

## Part B. Week 2 final: #31 `motorlevel_l_full` (emerging market)

1. **#12 `line_ensemble`** (the pre-probing model, public 0.16690): motor rows = `x20_tblend` predictions summed over perils;
   buildings and contents rows = `rt_var_b8_floor_corrected` predictions summed over perils.
2. **Motor level review**, using the training data only and no leaderboard probing. Buildings and contents are unchanged.
   - `F_BI`, motor bodily-injury severity: claims inflation of 4.5% a year (trend to 2025, and the mature 2021-23 years to base) x development x large-loss load (developed / emerging).
   - `F_NB`, motor new-business rows: the full actual-to-expected of new business in the emerging data / the new-business load already in the model.

In [ ]:
G_BI <- 0.045; SH23 <- 266 / 449
F_TREND <- 1 + G_BI; F_BASE <- 1 + G_BI * SH23
F_DEV <- (266 * 1.007 + 183 * 1.031) / 449
ld <- fread(find_one("loadings\\.csv$"))[line == "motor"]
F_LL <- ld[peril == "bodily_injury", developed_load / emerging_load]
F_BI <- F_TREND * F_BASE * F_DEV * F_LL
F_NB <- ld$nb_ae_rel[1] / ld$nb_load[1]
cat(sprintf("F_BI = %.4f (trend %.4f x to-base %.4f x development %.4f x large-loss %.4f), F_NB = %.4f\n",
            F_BI, F_TREND, F_BASE, F_DEV, F_LL, F_NB))

m <- as.data.table(read_parquet(find_one("x20_tblend.*pred_2025_peril\\.parquet$")))[line == "motor",
       .(row_id, peril, nb = new_business & !nb_factor, pred_B)]
p <- as.data.table(read_parquet(find_one("rt_var_b8_floor_corrected.*pred_2025_peril\\.parquet$")))[line != "motor"]
s12 <- rbind(m[, .(prediction = sum(pred_B)), by = row_id], p[, .(prediction = sum(pred_B)), by = row_id])
setorder(s12, row_id)

# new business only where it is not already a model factor (glass / hail has it as a factor)
k <- rep(1, nrow(m))
k <- k * fifelse(m$peril == "bodily_injury", F_BI, 1)
k <- k * fifelse(m$nb, F_NB, 1)
m[, pred_B := pred_B * k]
w2 <- copy(s12)[m[, .(p = sum(pred_B)), by = row_id], prediction := i.p, on = "row_id"]

im <- startsWith(s12$row_id, "M")
cat(sprintf("motor total #31 / #12 = %.4f; buildings and contents unchanged: %s\n",
            sum(w2$prediction[im]) / sum(s12$prediction[im]), identical(w2$prediction[!im], s12$prediction[!im])))

check_and_write(w2, 209063L, "90120c605cee98bb3545e0b7b4b9a128",
                "week2_submission_20261030_0905_week2_motorlevel_l_full\\.csv$", "submission_week2.csv")
w2[, .(policies = .N, mean_prediction = round(mean(prediction), 2)), by = .(line = line_of(row_id))]

## Output files

In [ ]:
file.info(file.path(OUT, c("submission_week1.csv", "submission_week2.csv")))[, "size", drop = FALSE]